# Maia Conversational Corpus Builder v3.0 — Notebook 1

**Objetivo:** construir e auditar um corpus de até **60.000 diálogos** (PT/EN/ES), sem inventar dados nem preencher cotas artificialmente. Este notebook é a **primeira etapa de ingestão e curadoria**, não o treinador.

- Ingestão de fontes públicas, com versões e licenças registradas.
- Separação **fail-closed** de fontes não comerciais ou de licença não verificada.
- Normalização de mensagens, filtros, deduplicação e divisão por grupo antes do treino.
- Relatórios de cobertura, rejeições, hashes e exportação JSONL.

**Importante:** a meta de 60 mil é um teto de composição, não um resultado garantido. Cotas insuficientes são informadas. OASST2 tem estrutura em árvore e exige reconstrução das cadeias; não concatenamos mensagens desconexas. O pipeline não usa geração sintética sem teacher e verificador configurados.

In [ ]:
%pip -q install datasets huggingface_hub pandas pyarrow tqdm langdetect nbformat

## 1. Configuração

Execute no Colab com acesso à internet. Por padrão, o notebook executa um **ensaio limitado**, sem tentar baixar todos os dados. Ative `FULL_BUILD=True` após conferir o relatório de amostra. Fontes sem licença/proveniência aprovada ficam em quarentena; **não altere essa regra para forçar o preenchimento**.

In [ ]:
from pathlib import Path
import os, json, re, hashlib, random, collections, datetime, unicodedata, itertools
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import HfApi

SEED = 20261120
random.seed(SEED)
FULL_BUILD = False
COMMERCIAL_ONLY = True
USE_DRIVE = True
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v3_0') if USE_DRIVE else Path('/content/maia_corpus_v3')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT.mkdir(parents=True, exist_ok=True)
OUT = ROOT / ('full' if FULL_BUILD else 'pilot')
OUT.mkdir(parents=True, exist_ok=True)

LANGS = ['pt','en','es']
TARGETS = {'conversation':7000,'comprehension':4000,'logic':4000,'instruction':3000,'uncertainty':2000}
MAX_PER_SOURCE = 100000 if FULL_BUILD else 300
MAX_RECORDS = 60000 if FULL_BUILD else 900
MIN_CHARS = 4
MAX_CHARS_PER_MESSAGE = 5000
MAX_MESSAGES = 14
# Licenças são hipóteses a verificar na revisão do manifesto. Nunca inferir permissão de derivados.
SOURCES = [
  {'id':'oasst2','repo':'OpenAssistant/oasst2','split':'train','kind':'oasst_tree','enabled':True,
   'license_review':'PENDING_UPSTREAM_REVIEW','declared_license':'apache-2.0'},
  {'id':'ultrachatbr','repo':'recogna-nlp/UltrachatBR','split':'train','kind':'chat_auto','enabled':True,
   'license_review':'PENDING_UPSTREAM_REVIEW','declared_license':'mit'},
  {'id':'aya','repo':'CohereLabs/aya_dataset','split':'train','kind':'prompt_completion','enabled':False,
   'license_review':'PENDING_UPSTREAM_REVIEW','declared_license':'unverified'},
]
# Somente revisão manual documentada autoriza exportação para treino.
APPROVED_SOURCE_IDS = set()  # ex.: {'oasst2'} APÓS revisão de origem e licença
print('Saída:', OUT, '| Full:', FULL_BUILD, '| Fontes aprovadas:', APPROVED_SOURCE_IDS)

## 2. Inspecionar fontes e metadados (antes de ingerir)

A disponibilidade, os campos e a licença de cada repositório podem mudar. Esta célula registra metadados atuais e mostra exemplos de esquema; erros ficam documentados.

In [ ]:
api = HfApi()
source_manifest = []
for src in SOURCES:
    item = dict(src)
    try:
        info = api.dataset_info(src['repo'])
        item['sha'] = info.sha
        item['card_license'] = (info.cardData or {}).get('license') if info.cardData else None
        item['siblings_count'] = len(info.siblings or [])
        item['fetch_status'] = 'ok'
    except Exception as e:
        item['fetch_status'] = 'error'
        item['error'] = str(e)[:500]
    source_manifest.append(item)
(OUT/'sources_manifest.json').write_text(json.dumps(source_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame(source_manifest)[['id','repo','declared_license','card_license','license_review','fetch_status']])

## 3. Normalização e validação

Apenas mensagens com papéis `user`/`assistant` são supervisionadas. `system` não é inventado. Para evitar contaminação entre traduções, agrupamentos posteriores usam origem + assinatura do conteúdo.

In [ ]:
def norm_lang(s):
    s=str(s or '').lower().replace('_','-')
    if s.startswith('pt'): return 'pt'
    if s.startswith('en'): return 'en'
    if s.startswith('es'): return 'es'
    return None

def clean_text(x):
    x = unicodedata.normalize('NFC', str(x or ''))
    x = x.replace('\x00','').replace('\r\n','\n')
    return re.sub(r'\n{4,}', '\n\n\n', x).strip()

def digest(x): return hashlib.sha256(x.encode('utf-8')).hexdigest()

def validate_messages(messages):
    if not isinstance(messages,list) or not (2 <= len(messages) <= MAX_MESSAGES): return 'invalid_turn_count'
    if messages[0]['role'] != 'user' or messages[-1]['role'] != 'assistant': return 'invalid_roles'
    for i,m in enumerate(messages):
        if m['role'] not in ('user','assistant'): return 'invalid_role'
        if i and m['role']==messages[i-1]['role']: return 'non_alternating_roles'
        t=m['content']
        if not (MIN_CHARS <= len(t) <= MAX_CHARS_PER_MESSAGE): return 'invalid_text_length'
        if re.search(r'(?i)(ignore previous instructions|system prompt|jailbreak)',t): return 'injection_pattern_review'
    return None

def make_record(src, messages, lang, raw_id, category='conversation', group=None):
    lang=norm_lang(lang)
    if lang not in LANGS: return None,'unsupported_language'
    messages=[{'role':m['role'],'content':clean_text(m['content'])} for m in messages]
    error=validate_messages(messages)
    if error:return None,error
    canonical=json.dumps(messages,ensure_ascii=False,sort_keys=True)
    uid=digest(src['id']+'|'+str(raw_id)+'|'+canonical)[:20]
    rec={'id':'maia_'+uid,'category':category,'subcategory':'source_import',
         'lang':lang,'source':src['repo'],'source_id':src['id'],'source_revision':src.get('sha'),
         'source_record_id':str(raw_id),'group_id':str(group or raw_id),
         'license_status':'approved' if src['id'] in APPROVED_SOURCE_IDS else 'quarantine',
         'quality_status':'unreviewed','messages':messages,'turns':len(messages),
         'content_sha256':digest(canonical)}
    return rec,None

def role_from(x):
    x=str(x or '').lower()
    if x in ('user','human','prompter'):return 'user'
    if x in ('assistant','gpt','bot'):return 'assistant'
    return None

def chat_messages(row):
    for key in ('messages','conversations','conversation','dialogue','chat'):
        val=row.get(key)
        if isinstance(val,list):
            out=[]
            for m in val:
                if not isinstance(m,dict):return None
                role=role_from(m.get('role',m.get('from',m.get('speaker'))))
                content=m.get('content',m.get('value',m.get('text')))
                if role and isinstance(content,str):out.append({'role':role,'content':content})
            return out or None
    return None

def pair_messages(row):
    prompt=row.get('inputs',row.get('prompt',row.get('instruction',row.get('question'))))
    answer=row.get('targets',row.get('completion',row.get('output',row.get('answer'))))
    if isinstance(prompt,str) and isinstance(answer,str):
        return [{'role':'user','content':prompt},{'role':'assistant','content':answer}]
    return None

## 4. Ingestão — estrutura real dos datasets

OASST2 é reconstruído a partir de `message_id` e `parent_id`. Os outros formatos passam por detecção de esquema; **esquemas desconhecidos são rejeitados**, não adivinhados. O notebook limita o volume para evitar estouro de RAM.

In [ ]:
def read_rows(src):
    try:
        ds=load_dataset(src['repo'],split=src['split'],streaming=True,trust_remote_code=False)
        for i,row in enumerate(itertools.islice(ds,MAX_PER_SOURCE)):
            yield i,row
    except Exception as e:
        errors.append({'source':src['id'],'reason':'load_error','detail':str(e)[:400]})

def parse_oasst(src, rows):
    nodes={}
    for i,r in rows:
        mid=r.get('message_id')
        if mid is not None: nodes[mid]=r
    for mid,r in nodes.items():
        if role_from(r.get('role'))!='assistant':continue
        chain=[]; visited=set(); cur=r
        while cur is not None and len(chain)<MAX_MESSAGES and cur.get('message_id') not in visited:
            visited.add(cur.get('message_id'))
            role=role_from(cur.get('role'))
            if role is None:break
            chain.append({'role':role,'content':cur.get('text','')})
            cur=nodes.get(cur.get('parent_id'))
        chain.reverse()
        if len(chain)<2:continue
        yield mid,chain,r.get('lang'),r.get('message_tree_id') or mid

errors=[]; records=[]; source_stats=collections.Counter()
for src in source_manifest:
    if not src['enabled']:continue
    print('Processando',src['repo'])
    rows=list(read_rows(src)) if src['kind']=='oasst_tree' else None
    if src['kind']=='oasst_tree':
        parsed=parse_oasst(src,rows)
    else:
        def iterator():
            for i,row in read_rows(src):
                msg=chat_messages(row) if src['kind']=='chat_auto' else pair_messages(row)
                yield i,msg,row.get('lang',row.get('language')),row.get('id',i)
        parsed=iterator()
    for rid,msg,lang,group in parsed:
        if msg is None:
            source_stats[(src['id'],'schema_unrecognized')]+=1
            continue
        rec,err=make_record(src,msg,lang,rid,group=group)
        if err:source_stats[(src['id'],err)]+=1
        else:
            records.append(rec);source_stats[(src['id'],'accepted_schema')]+=1
        if len(records)>=MAX_RECORDS:break
    if len(records)>=MAX_RECORDS:break
print('Registros normalizados:',len(records),'| erros:',len(errors))
display(pd.DataFrame([{'source':s,'status':k,'n':v} for (s,k),v in source_stats.items()]).sort_values('n',ascending=False).head(30))
if errors:display(pd.DataFrame(errors).head(10))
if records:display(pd.DataFrame([{k:r[k] for k in ('id','lang','category','source_id','turns','license_status')} for r in records]).head())

## 5. Deduplicação e isolamento por família

Deduplicação exata e normalizada; similaridade semântica e LLM-as-a-judge **não são simulados**. Uma fase posterior poderá adicioná-los. `group_id` é preservado no split para impedir vazamento dentro de uma árvore OASST.

In [ ]:
def fingerprint(messages):
    text=' '.join(m['role']+': '+m['content'] for m in messages)
    text=unicodedata.normalize('NFKC',text).casefold()
    return digest(re.sub(r'\s+',' ',text).strip())
seen=set(); unique=[]; rejects=[]
for r in records:
    fp=fingerprint(r['messages'])
    if fp in seen:
        rejects.append({'id':r['id'],'reason':'duplicate_normalized','source_id':r['source_id']})
    else:
        seen.add(fp);unique.append(r)
print('Únicos:',len(unique),'| duplicados:',len(rejects))
# Por segurança, apenas fontes aprovadas podem entrar no dataset de treino.
approved=[r for r in unique if r['license_status']=='approved']
quarantine=[r for r in unique if r['license_status']!='approved']
print('Aprovados:',len(approved),'| quarentena:',len(quarantine))

## 6. Seleção e splits determinísticos

As cotas são por **categoria × idioma**. Este primeiro notebook não reclassifica artificialmente exemplos de conversação em lógica/instruções. Se faltarem categorias, o relatório mostra o déficit para a próxima fase de geração/curadoria. A divisão é feita por grupo de origem.

In [ ]:
def group_split(r):
    # Mesma família sempre no mesmo split, independentemente de tradução ou variante.
    key=r['source_id']+'|'+r['group_id']
    p=int(digest(key)[:8],16)/0xffffffff
    return 'train' if p<0.90 else ('validation' if p<0.95 else 'test')

selected=[]; count=collections.Counter()
for r in sorted(approved,key=lambda x:digest(str(SEED)+'|'+x['id'])):
    key=(r['category'],r['lang'])
    if count[key] < TARGETS.get(r['category'],0):
        r=dict(r,split=group_split(r))
        selected.append(r);count[key]+=1
quota=[]
for cat,target in TARGETS.items():
    for lang in LANGS:
        got=count[(cat,lang)]
        quota.append({'category':cat,'lang':lang,'target':target,'obtained':got,'missing':target-got})
coverage=pd.DataFrame(quota)
display(coverage)
print('Total selecionado:',len(selected),'de 60000; faltam:',int(coverage.missing.sum()))

## 7. Auditoria de conteúdo e exportação

O arquivo de quarentena é separado e **não é treino**. Exigir auditoria humana de qualidade e licença antes de liberar qualquer fonte. O notebook escreve hashes SHA-256 e não declara sucesso quando há déficit.

In [ ]:
def write_jsonl(path,items):
    with open(path,'w',encoding='utf-8') as f:
        for x in items:f.write(json.dumps(x,ensure_ascii=False)+'\n')

for split in ('train','validation','test'):
    write_jsonl(OUT/f'{split}.jsonl',(r for r in selected if r['split']==split))
write_jsonl(OUT/'quarantine.jsonl',quarantine)
write_jsonl(OUT/'rejected.jsonl',rejects)
coverage.to_csv(OUT/'coverage.csv',index=False)
(OUT/'ingest_errors.json').write_text(json.dumps(errors,indent=2,ensure_ascii=False),encoding='utf-8')
config={'seed':SEED,'full_build':FULL_BUILD,'commercial_only':COMMERCIAL_ONLY,
        'approved_source_ids':sorted(APPROVED_SOURCE_IDS),'max_per_source':MAX_PER_SOURCE,
        'target_total':60000,'selected_total':len(selected),'quarantine_total':len(quarantine),
        'missing_total':int(coverage.missing.sum()),'status':'TARGET_MET' if len(selected)==60000 else 'INCOMPLETE',
        'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
(OUT/'build_report.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
files=[]
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name!='sha256_manifest.json':
        files.append({'file':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
(OUT/'sha256_manifest.json').write_text(json.dumps(files,indent=2),encoding='utf-8')
print(json.dumps(config,indent=2,ensure_ascii=False))
display(pd.DataFrame(files))

## 8. Bloqueios de liberação para SFT

Este notebook **não deve ser considerado um construtor completo de 60 mil exemplos** até que fontes, licenças e categorias estejam preenchidas e auditadas. As etapas seguintes serão: (1) selecionar fontes EN/ES e de compreensão; (2) gerador sintético verificado para lógica/instruções/incerteza; (3) curadoria humana/automática; (4) deduplicação semântica e revisão de vazamento; (5) tokenizer Maia Lite, contagem de tokens e máscara de loss; (6) congelar manifesto e exportar o corpus final. A validação automática não substitui revisão de amostras nem auditoria de direitos.

In [ ]:
assert len({r['id'] for r in selected})==len(selected)
assert all(r['license_status']=='approved' for r in selected)
assert all(validate_messages(r['messages']) is None for r in selected)
groups=collections.defaultdict(set)
for r in selected:groups[(r['source_id'],r['group_id'])].add(r['split'])
assert all(len(s)==1 for s in groups.values()), 'Vazamento de grupo entre splits'
print('Verificações estruturais aprovadas.')
if config['status']!='TARGET_MET':
    print('ATENÇÃO: corpus incompleto. Não iniciar SFT a partir destes arquivos.')